In [2]:
import pandas as pd
import numpy as np

file_name = "Cleaned_Marketplace_Data.xlsx"

df = pd.read_excel(file_name)

print("Dataset loaded successfully.")

df["Order_Date"] = pd.to_datetime(
    df["Order_Date"],
    errors="coerce"
)

print("\n----- DATASET OVERVIEW -----")
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

print("\n----- DATA PREVIEW -----")
display(df.head())


# ------------------------------------------------------------
# Overall Marketplace Metrics
# ------------------------------------------------------------

total_customers = df["Customer_ID"].nunique()

total_orders = df["Order_ID"].nunique()

total_sales = df["Sales"].sum()

total_quantity = df["Quantity"].sum()

average_order_value = (
    total_sales / total_orders
    if total_orders > 0 else 0
)

average_quantity_per_order = (
    total_quantity / total_orders
    if total_orders > 0 else 0
)

average_orders_per_customer = (
    total_orders / total_customers
    if total_customers > 0 else 0
)

repeat_customers = (
    df.groupby("Customer_ID")["Order_ID"]
      .nunique()
)

repeat_customer_count = (
    repeat_customers[repeat_customers > 1]
    .count()
)

repeat_customer_rate = (
    repeat_customer_count / total_customers * 100
    if total_customers > 0 else 0
)

overall_metrics = pd.DataFrame({
    "Metric": [
        "Total Customers",
        "Total Orders",
        "Total Sales",
        "Total Quantity",
        "Average Order Value",
        "Average Quantity per Order",
        "Average Orders per Customer",
        "Repeat Customers",
        "Repeat Customer Rate"
    ],
    "Value": [
        total_customers,
        total_orders,
        round(total_sales, 2),
        total_quantity,
        round(average_order_value, 2),
        round(average_quantity_per_order, 2),
        round(average_orders_per_customer, 2),
        repeat_customer_count,
        round(repeat_customer_rate, 2)
    ]
})

print("\n----- OVERALL MARKETPLACE METRICS -----")
display(overall_metrics)


# ------------------------------------------------------------
# Daily Marketplace Activity
# ------------------------------------------------------------

daily_activity = (
    df.groupby("Order_Date")
      .agg(
          Orders=("Order_ID", "nunique"),
          Customers=("Customer_ID", "nunique"),
          Sales=("Sales", "sum"),
          Quantity=("Quantity", "sum")
      )
      .reset_index()
)

daily_activity["Average_Order_Value"] = (
    daily_activity["Sales"] /
    daily_activity["Orders"]
)

daily_activity["Average_Order_Value"] = (
    daily_activity["Average_Order_Value"].round(2)
)

print("\n----- DAILY MARKETPLACE ACTIVITY -----")
display(daily_activity.head(10))


# ------------------------------------------------------------
# Monthly Marketplace Activity
# ------------------------------------------------------------

df["Month"] = df["Order_Date"].dt.to_period("M").astype(str)

monthly_activity = (
    df.groupby("Month")
      .agg(
          Orders=("Order_ID", "nunique"),
          Customers=("Customer_ID", "nunique"),
          Sales=("Sales", "sum"),
          Quantity=("Quantity", "sum")
      )
      .reset_index()
)

monthly_activity["Average_Order_Value"] = (
    monthly_activity["Sales"] /
    monthly_activity["Orders"]
)

monthly_activity["Average_Order_Value"] = (
    monthly_activity["Average_Order_Value"].round(2)
)

print("\n----- MONTHLY MARKETPLACE ACTIVITY -----")
display(monthly_activity)


# ------------------------------------------------------------
# Customer Activity
# ------------------------------------------------------------

customer_activity = (
    df.groupby("Customer_ID")
      .agg(
          Orders=("Order_ID", "nunique"),
          Sales=("Sales", "sum"),
          Quantity=("Quantity", "sum"),
          First_Order_Date=("Order_Date", "min"),
          Last_Order_Date=("Order_Date", "max")
      )
      .reset_index()
)

customer_activity["Average_Order_Value"] = (
    customer_activity["Sales"] /
    customer_activity["Orders"]
)

customer_activity["Average_Order_Value"] = (
    customer_activity["Average_Order_Value"].round(2)
)

print("\n----- CUSTOMER ACTIVITY -----")
display(customer_activity.head(10))


# ------------------------------------------------------------
# Repeat Customer Analysis
# ------------------------------------------------------------

customer_activity["Customer_Type"] = np.where(
    customer_activity["Orders"] > 1,
    "Repeat Customer",
    "One-Time Customer"
)

customer_type_summary = (
    customer_activity
    .groupby("Customer_Type")
    .agg(
        Customer_Count=("Customer_ID", "nunique"),
        Total_Orders=("Orders", "sum"),
        Total_Sales=("Sales", "sum")
    )
    .reset_index()
)

customer_type_summary["Sales_Percentage"] = (
    customer_type_summary["Total_Sales"] /
    total_sales * 100
).round(2)

print("\n----- CUSTOMER TYPE SUMMARY -----")
display(customer_type_summary)


# ------------------------------------------------------------
# Product Category Performance
# ------------------------------------------------------------

category_performance = (
    df.groupby("Product_Category")
      .agg(
          Orders=("Order_ID", "nunique"),
          Customers=("Customer_ID", "nunique"),
          Quantity=("Quantity", "sum"),
          Sales=("Sales", "sum")
      )
      .reset_index()
)

category_performance["Average_Order_Value"] = (
    category_performance["Sales"] /
    category_performance["Orders"]
)

category_performance["Sales_Percentage"] = (
    category_performance["Sales"] /
    total_sales * 100
)

category_performance["Average_Order_Value"] = (
    category_performance["Average_Order_Value"].round(2)
)

category_performance["Sales_Percentage"] = (
    category_performance["Sales_Percentage"].round(2)
)

category_performance = category_performance.sort_values(
    "Sales",
    ascending=False
)

print("\n----- CATEGORY PERFORMANCE -----")
display(category_performance)


# ------------------------------------------------------------
# Top Customers
# ------------------------------------------------------------

top_customers = (
    customer_activity
    .sort_values(
        "Sales",
        ascending=False
    )
    .head(10)
)

print("\n----- TOP 10 CUSTOMERS -----")
display(top_customers)


# ------------------------------------------------------------
# Top Orders
# ------------------------------------------------------------

top_orders = (
    df[
        [
            "Order_ID",
            "Customer_ID",
            "Order_Date",
            "Product_Category",
            "Quantity",
            "Unit_Price",
            "Sales"
        ]
    ]
    .sort_values(
        "Sales",
        ascending=False
    )
    .head(10)
)

print("\n----- TOP 10 ORDERS -----")
display(top_orders)


# ------------------------------------------------------------
# Order Value Analysis
# ------------------------------------------------------------

order_summary = (
    df.groupby("Order_ID")
      .agg(
          Customer_ID=("Customer_ID", "first"),
          Order_Date=("Order_Date", "first"),
          Quantity=("Quantity", "sum"),
          Sales=("Sales", "sum")
      )
      .reset_index()
)

order_value_analysis = pd.DataFrame({
    "Metric": [
        "Minimum Order Value",
        "Average Order Value",
        "Median Order Value",
        "Maximum Order Value"
    ],
    "Value": [
        round(order_summary["Sales"].min(), 2),
        round(order_summary["Sales"].mean(), 2),
        round(order_summary["Sales"].median(), 2),
        round(order_summary["Sales"].max(), 2)
    ]
})

print("\n----- ORDER VALUE ANALYSIS -----")
display(order_value_analysis)


# ------------------------------------------------------------
# Marketplace Health Summary
# ------------------------------------------------------------

marketplace_health = pd.DataFrame({
    "Metric": [
        "Total Customers",
        "Total Orders",
        "Total Sales",
        "Total Quantity",
        "Average Order Value",
        "Average Orders per Customer",
        "Repeat Customer Rate",
        "Highest Sales Category"
    ],
    "Value": [
        total_customers,
        total_orders,
        round(total_sales, 2),
        total_quantity,
        round(average_order_value, 2),
        round(average_orders_per_customer, 2),
        round(repeat_customer_rate, 2),
        category_performance.iloc[0]["Product_Category"]
    ]
})

print("\n----- MARKETPLACE HEALTH SUMMARY -----")
display(marketplace_health)


# ------------------------------------------------------------
# Final Output File
# ------------------------------------------------------------

output_file = "Marketplace_Analysis_Result.xlsx"

with pd.ExcelWriter(
    output_file,
    engine="openpyxl"
) as writer:

    overall_metrics.to_excel(
        writer,
        sheet_name="Overall Metrics",
        index=False
    )

    daily_activity.to_excel(
        writer,
        sheet_name="Daily Activity",
        index=False
    )

    monthly_activity.to_excel(
        writer,
        sheet_name="Monthly Activity",
        index=False
    )

    customer_activity.to_excel(
        writer,
        sheet_name="Customer Activity",
        index=False
    )

    customer_type_summary.to_excel(
        writer,
        sheet_name="Customer Types",
        index=False
    )

    category_performance.to_excel(
        writer,
        sheet_name="Category Performance",
        index=False
    )

    top_customers.to_excel(
        writer,
        sheet_name="Top Customers",
        index=False
    )

    top_orders.to_excel(
        writer,
        sheet_name="Top Orders",
        index=False
    )

    order_value_analysis.to_excel(
        writer,
        sheet_name="Order Value",
        index=False
    )

    marketplace_health.to_excel(
        writer,
        sheet_name="Marketplace Health",
        index=False
    )


print("\n========================================")
print("ANALYSIS COMPLETED SUCCESSFULLY")
print("========================================")

print("Output File:", output_file)

print("\nTotal Customers:", total_customers)
print("Total Orders:", total_orders)
print("Total Sales:", round(total_sales, 2))
print("Repeat Customer Rate:", round(repeat_customer_rate, 2), "%")

print("\nAll analysis results are stored in ONE Excel file.")

Dataset loaded successfully.

----- DATASET OVERVIEW -----
Rows: 983
Columns: 7

----- DATA PREVIEW -----


,Customer_ID,Order_ID,Order_Date,Product_Category,Quantity,Unit_Price,Sales
0,CUST0103,ORD00001,2025-08-24,Furniture,4,9571.20,38284.80
1,CUST0180,ORD00002,2025-03-24,Electronics,3,11596.09,34788.27
2,CUST0015,ORD00004,2025-04-11,Office Supplies,1,12905.51,12905.51
3,CUST0107,ORD00005,2025-01-06,Electronics,4,2912.44,11649.76
4,CUST0072,ORD00006,2025-06-29,Furniture,5,8348.23,41741.15



----- OVERALL MARKETPLACE METRICS -----


,Metric,Value
0,Total Customers,198.00
1,Total Orders,983.00
2,Total Sales,23921205.29
3,Total Quantity,2992.00
4,Average Order Value,24334.90
5,Average Quantity per Order,3.04
6,Average Orders per Customer,4.96
7,Repeat Customers,191.00
8,Repeat Customer Rate,96.46



----- DAILY MARKETPLACE ACTIVITY -----


...


----- ORDER VALUE ANALYSIS -----


,Metric,Value
0,Minimum Order Value,249.23
1,Average Order Value,24334.90
2,Median Order Value,21367.35
3,Maximum Order Value,74722.45



----- MARKETPLACE HEALTH SUMMARY -----


,Metric,Value
0,Total Customers,198
1,Total Orders,983
2,Total Sales,23921205.29
3,Total Quantity,2992
4,Average Order Value,24334.9
5,Average Orders per Customer,4.96
6,Repeat Customer Rate,96.46
7,Highest Sales Category,Electronics



ANALYSIS COMPLETED SUCCESSFULLY
Output File: Marketplace_Analysis_Result.xlsx

Total Customers: 198
Total Orders: 983
Total Sales: 23921205.29
Repeat Customer Rate: 96.46 %

All analysis results are stored in ONE Excel file.
